In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm==0.8.5.post1

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm==0.8.5.post1
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [ ]:
# Model Inference Notebook
# Load and compare three fine-tuned scholarly models

# Install required packages
!pip install -q unsloth transformers torch

import torch
from transformers import AutoTokenizer, TextStreamer
from unsloth import FastModel, FastLanguageModel
import time
import pandas as pd

In [ ]:
# SECTION 1: LOAD MODELS WITH THEIR ADAPTERS
print("Loading models... This may take several minutes depending on your hardware.")

# 1. PHI-4 SCHOLAR
print("\n[1/3] Loading Phi-4 Scholar...")
phi4_model, phi4_tokenizer = FastModel.from_pretrained(
    model_name="unsloth/Phi-4",
    max_seq_length=2048,
    load_in_4bit=True,
)
# Load LoRA adapter
phi4_model = FastModel.from_adapter(
    phi4_model,
    adapter_name_or_path="JunaidSadiq/Phi4_scholar"
)
# Set for inference
FastModel.for_inference(phi4_model)
print("✓ Phi-4 Scholar loaded successfully")

# 2. QWEN RESEARCH ASSISTANT
print("\n[2/3] Loading Qwen Research Assistant...")
qwen_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Base",
    max_seq_length=2048,
    load_in_4bit=True,
)
# Load LoRA adapter
qwen_model = FastLanguageModel.from_adapter(
    qwen_model,
    adapter_name_or_path="JunaidSadiq/Qwen_ResearchAssistant"
)
# Set for inference
FastLanguageModel.for_inference(qwen_model)
print("✓ Qwen Research Assistant loaded successfully")

# 3. GEMMA SCHOLAR
print("\n[3/3] Loading Gemma Scholar...")
gemma_model, gemma_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
gemma_model = FastLanguageModel.from_adapter(
    gemma_model,
    adapter_name_or_path = "JunaidSadiq/gemma-scholar-1b"
)
# Set for inference - this was missing
FastLanguageModel.for_inference(gemma_model)
print("✓ Gemma Scholar loaded successfully")

In [ ]:
# SECTION 2: PREPARE TEST QUERIES
# Define 10 academic/research queries to test all models

academic_queries = [
    "Explain the concept of deep learning to a beginner",
    "What are the main approaches to natural language processing?",
    "Summarize the key findings from recent research on large language models",
    "How does transfer learning work in machine learning?",
    "What is the significance of the transformer architecture in AI?",
    "Explain the concept of game theory in economics",
    "What are the ethical considerations in AI research?",
    "Describe the peer review process in academic publishing",
    "What is the reproducibility crisis in science?",
    "How do citation metrics impact academic research?"
]

# Create a function to run inference and measure response time
def run_model_inference(model, tokenizer, query, model_name, max_tokens=512):
    # Format based on model type
    if "phi" in model_name.lower():
        # Phi-4 format
        messages = [{"role": "user", "content": query}]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    elif "qwen" in model_name.lower():
        # Qwen format - ChatML
        input_text = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
    else:
        # Gemma format
        messages = [{"role": "user", "content": query}]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    
    # Capture output in a variable instead of streaming
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    # Measure time
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            top_k=40,
        )
    
    end_time = time.time()
    
    # Decode the output, skipping the input prompt
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    return {
        "response": response,
        "time_taken": round(end_time - start_time, 2)
    }

In [ ]:
# SECTION 3: RUN INFERENCE ON PHI-4 SCHOLAR
print("Running inference on Phi-4 Scholar...\n" + "="*50)

phi4_results = []

for i, query in enumerate(academic_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("-" * 50)
    
    result = run_model_inference(phi4_model, phi4_tokenizer, query, "phi4")
    
    print(f"Response ({result['time_taken']} seconds):")
    print(result['response'])
    print("=" * 50)
    
    phi4_results.append({
        "query": query,
        "response": result['response'],
        "time_taken": result['time_taken']
    })
    
    # Free up memory
    torch.cuda.empty_cache()

print("\nPhi-4 Scholar inference completed!")

In [ ]:
# SECTION 4: RUN INFERENCE ON QWEN RESEARCH ASSISTANT
print("Running inference on Qwen Research Assistant...\n" + "="*50)

qwen_results = []

for i, query in enumerate(academic_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("-" * 50)
    
    result = run_model_inference(qwen_model, qwen_tokenizer, query, "qwen")
    
    print(f"Response ({result['time_taken']} seconds):")
    print(result['response'])
    print("=" * 50)
    
    qwen_results.append({
        "query": query,
        "response": result['response'],
        "time_taken": result['time_taken']
    })
    
    # Free up memory
    torch.cuda.empty_cache()

print("\nQwen Research Assistant inference completed!")

In [ ]:
# SECTION 5: RUN INFERENCE ON GEMMA SCHOLAR
print("Running inference on Gemma Scholar...\n" + "="*50)

gemma_results = []

for i, query in enumerate(academic_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("-" * 50)
    
    result = run_model_inference(gemma_model, gemma_tokenizer, query, "gemma")
    
    print(f"Response ({result['time_taken']} seconds):")
    print(result['response'])
    print("=" * 50)
    
    gemma_results.append({
        "query": query,
        "response": result['response'],
        "time_taken": result['time_taken']
    })
    
    # Free up memory
    torch.cuda.empty_cache()

print("\nGemma Scholar inference completed!")

In [ ]:
# SECTION 6: ANALYZE AND COMPARE PERFORMANCE
print("Comparing model performance metrics...\n")

# Create performance comparison dataframe
performance_df = pd.DataFrame({
    'Query': academic_queries,
    'Phi4_Time': [r['time_taken'] for r in phi4_results],
    'Qwen_Time': [r['time_taken'] for r in qwen_results],
    'Gemma_Time': [r['time_taken'] for r in gemma_results]
})

# Display performance metrics
print("Response Time Comparison (seconds):")
print(performance_df[['Query', 'Phi4_Time', 'Qwen_Time', 'Gemma_Time']])

# Calculate averages
print("\nAverage Response Times:")
print(f"Phi-4 Scholar: {performance_df['Phi4_Time'].mean():.2f} seconds")
print(f"Qwen Research Assistant: {performance_df['Qwen_Time'].mean():.2f} seconds")
print(f"Gemma Scholar: {performance_df['Gemma_Time'].mean():.2f} seconds")

# Create a bar chart
import matplotlib.pyplot as plt

# Set up the figure
plt.figure(figsize=(12, 6))
avg_times = [
    performance_df['Phi4_Time'].mean(),
    performance_df['Qwen_Time'].mean(),
    performance_df['Gemma_Time'].mean()
]
models = ['Phi-4 Scholar', 'Qwen Research Assistant', 'Gemma Scholar']
plt.bar(models, avg_times, color=['#4285F4', '#DB4437', '#F4B400'])
plt.title('Average Response Time by Model')
plt.ylabel('Time (seconds)')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, v in enumerate(avg_times):
    plt.text(i, v + 0.1, f'{v:.2f}s', ha='center')

plt.tight_layout()
plt.show()

print("\nAnalysis complete! You can now compare the quality of responses manually.")

In [ ]:
# OPTIONAL: Save all responses to file for further analysis
import json

# Combine all results
all_results = {
    "phi4_scholar": phi4_results,
    "qwen_research_assistant": qwen_results,
    "gemma_scholar": gemma_results
}

# Save to JSON file
with open('model_comparison_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Results saved to 'model_comparison_results.json'")